**Scrollytelling Study: Where Does Episode-of-Care Spending Diverge?**

Core question:
How does a facility's episode-of-care spending differ from its state and national benchmarks—and when in the episode do those differences emerge?



In [ ]:
!pip -q install plotly ipywidgets

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.4 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv("Medicare_Hospital_Spending_by_Claim.csv")
df.head(10)

,Facility Name,Facility ID,State,Period,Claim Type,Avg Spndg Per EP Hospital,Avg Spndg Per EP State,Avg Spndg Per EP National,Percent of Spndg Hospital,Percent of Spndg State,Percent of Spndg National,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Home Health Agency,25,22,19,0.09%,0.08%,0.07%,01/01/2024,12/31/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Hospice,2,2,1,0.01%,0.01%,0.00%,01/01/2024,12/31/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Inpatient,28,45,46,0.10%,0.17%,0.17%,01/01/2024,12/31/2024
3,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Outpatient,116,135,208,0.43%,0.52%,0.76%,01/01/2024,12/31/2024
4,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Skilled Nursing Facility,9,14,17,0.03%,0.05%,0.06%,01/01/2024,12/31/2024
5,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Durable Medical Equipment,6,11,12,0.02%,0.04%,0.04%,01/01/2024,12/31/2024
6,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,1 to 3 days Prior to Index Hospital Admission,Carrier,878,760,785,3.25%,2.90%,2.86%,01/01/2024,12/31/2024
7,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,During Index Hospital Admission,Home Health Agency,0,0,0,0.00%,0.00%,0.00%,01/01/2024,12/31/2024
8,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,During Index Hospital Admission,Hospice,0,0,0,0.00%,0.00%,0.00%,01/01/2024,12/31/2024
9,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,During Index Hospital Admission,Inpatient,12086,12072,12477,44.80%,46.07%,45.41%,01/01/2024,12/31/2024


In [ ]:
# ------------------------------------------------------------
# CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = [c.strip() for c in df.columns]


# ------------------------------------------------------------
# NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_cols = [
    "Avg Spndg Per EP Hospital",
    "Avg Spndg Per EP State",
    "Avg Spndg Per EP National",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)


# ------------------------------------------------------------
# PERCENT COLUMNS
# ------------------------------------------------------------

percent_cols = [
    "Percent of Spndg Hospital",
    "Percent of Spndg State",
    "Percent of Spndg National",
]

for col in percent_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)


# ------------------------------------------------------------
# FACILITY ID
# ------------------------------------------------------------

df["Facility ID"] = (
    df["Facility ID"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# DATES
# ------------------------------------------------------------

df["Start Date"] = pd.to_datetime(
    df["Start Date"],
    errors="coerce"
)

df["End Date"] = pd.to_datetime(
    df["End Date"],
    errors="coerce"
)


# ------------------------------------------------------------
# NORMALIZE PERIOD
# ------------------------------------------------------------

def normalize_period(x):

    x = str(x).lower().strip()

    if "prior" in x:
        return "Before admission"

    if "during" in x:
        return "Index admission"

    if "after discharge" in x:
        return "After discharge"

    if "complete episode" in x:
        return "Complete episode"

    return x


df["Phase"] = df["Period"].apply(normalize_period)


# ------------------------------------------------------------
# CLAIM TYPE
# ------------------------------------------------------------

df["Claim Type"] = df["Claim Type"].str.strip()


print("Prepared dataset:")
print(df.shape)

display(
    df[
        [
            "Facility Name",
            "Facility ID",
            "State",
            "Phase",
            "Claim Type",
            "Avg Spndg Per EP Hospital",
            "Avg Spndg Per EP State",
            "Avg Spndg Per EP National",
        ]
    ].head(10)
)

Prepared dataset:
(63646, 14)


,Facility Name,Facility ID,State,Phase,Claim Type,Avg Spndg Per EP Hospital,Avg Spndg Per EP State,Avg Spndg Per EP National
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Home Health Agency,25,22,19
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Hospice,2,2,1
2,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Inpatient,28,45,46
3,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Outpatient,116,135,208
4,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Skilled Nursing Facility,9,14,17
5,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Durable Medical Equipment,6,11,12
6,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Before admission,Carrier,878,760,785
7,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Index admission,Home Health Agency,0,0,0
8,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Index admission,Hospice,0,0,0
9,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,Index admission,Inpatient,12086,12072,12477


In [ ]:
# ============================================================
# ANALYTICAL AGGREGATIONS
# ============================================================

GROUP_COLS = [
    "Facility ID",
    "Facility Name",
    "State",
    "Phase",
    "Claim Type",
]


agg = (
    df
    .groupby(GROUP_COLS, as_index=False)
    .agg(
        Hospital=(
            "Avg Spndg Per EP Hospital",
            "sum"
        ),
        StateBenchmark=(
            "Avg Spndg Per EP State",
            "sum"
        ),
        NationalBenchmark=(
            "Avg Spndg Per EP National",
            "sum"
        ),
    )
)


agg["Vs State"] = (
    agg["Hospital"]
    - agg["StateBenchmark"]
)

agg["Vs National"] = (
    agg["Hospital"]
    - agg["NationalBenchmark"]
)

agg["Vs State %"] = np.where(
    agg["StateBenchmark"] != 0,
    (
        agg["Hospital"]
        - agg["StateBenchmark"]
    )
    / agg["StateBenchmark"]
    * 100,
    np.nan,
)

agg["Vs National %"] = np.where(
    agg["NationalBenchmark"] != 0,
    (
        agg["Hospital"]
        - agg["NationalBenchmark"]
    )
    / agg["NationalBenchmark"]
    * 100,
    np.nan,
)


print(
    f"Analytical table: "
    f"{len(agg):,} rows × {len(agg.columns)} columns"
)

Analytical table: 63,646 rows × 12 columns


In [ ]:
# ============================================================
# HELPERS
# ============================================================

COLORS = {
    "hospital": "#174A5B",
    "state": "#7A8793",
    "national": "#C7CED4",
    "positive": "#D97706",
    "negative": "#168A72",
}


def money(x):
    return f"${x:,.0f}"


def get_facility(facility_id):

    return agg[
        agg["Facility ID"] == str(facility_id)
    ].copy()


def get_complete_episode(facility_id):

    x = get_facility(facility_id)

    return x[
        x["Phase"] == "Complete episode"
    ].copy()


def get_phase(facility_id, phase):

    return agg[
        (agg["Facility ID"] == str(facility_id))
        & (agg["Phase"] == phase)
    ].copy()


def facility_info(facility_id):

    x = agg[
        agg["Facility ID"] == str(facility_id)
    ]

    if x.empty:
        return None

    return {
        "name": x["Facility Name"].iloc[0],
        "state": x["State"].iloc[0],
        "id": str(facility_id),
    }

In [ ]:
def chart_total_episode(facility_id):

    x = get_complete_episode(facility_id)

    if x.empty:
        print("No complete episode row found.")
        return

    # There should normally be one Total row.
    x = x[x["Claim Type"] == "Total"]

    if x.empty:
        print("No Total row found.")
        return

    row = x.iloc[0]

    values = [
        row["Hospital"],
        row["StateBenchmark"],
        row["NationalBenchmark"],
    ]

    labels = [
        "Facility",
        "State",
        "National",
    ]

    fig = go.Figure(
        go.Bar(
            x=labels,
            y=values,
            marker_color=[
                COLORS["hospital"],
                COLORS["state"],
                COLORS["national"],
            ],
            text=[
                money(v)
                for v in values
            ],
            textposition="outside",
        )
    )

    fig.update_layout(
        title="Complete episode spending",
        yaxis_title="Average spending per episode",
        template="plotly_white",
        height=450,
        showlegend=False,
    )

    fig.show()

In [ ]:
facility_id = "10001"

chart_total_episode(facility_id)

In [ ]:
def chart_episode_phases(facility_id):

    phases = [
        "Before admission",
        "Index admission",
        "After discharge",
    ]

    x = get_facility(facility_id)

    x = x[
        x["Phase"].isin(phases)
    ]

    x = (
        x
        .groupby("Phase", as_index=False)
        .agg(
            Hospital=("Hospital", "sum"),
            State=("StateBenchmark", "sum"),
            National=("NationalBenchmark", "sum"),
        )
    )

    x["Phase"] = pd.Categorical(
        x["Phase"],
        categories=phases,
        ordered=True,
    )

    x = x.sort_values("Phase")

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=x["Phase"],
            y=x["Hospital"],
            name="Facility",
            marker_color=COLORS["hospital"],
        )
    )

    fig.add_trace(
        go.Bar(
            x=x["Phase"],
            y=x["State"],
            name="State",
            marker_color=COLORS["state"],
        )
    )

    fig.add_trace(
        go.Bar(
            x=x["Phase"],
            y=x["National"],
            name="National",
            marker_color=COLORS["national"],
        )
    )

    fig.update_layout(
        title="Spending across the episode",
        barmode="group",
        yaxis_title="Average spending per episode",
        template="plotly_white",
        height=500,
    )

    fig.show()

In [ ]:
chart_episode_phases("10001")

In [ ]:
def chart_claim_mix(
    facility_id,
    phase="After discharge"
):

    x = get_phase(
        facility_id,
        phase
    )

    x = x[
        x["Claim Type"] != "Total"
    ]

    if x.empty:
        print("No data.")
        return

    x = (
        x
        .groupby("Claim Type", as_index=False)
        .agg(
            Facility=("Hospital", "sum"),
            State=("StateBenchmark", "sum"),
            National=("NationalBenchmark", "sum"),
        )
        .sort_values("Facility")
    )

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            y=x["Claim Type"],
            x=x["Facility"],
            name="Facility",
            orientation="h",
            marker_color=COLORS["hospital"],
        )
    )

    fig.add_trace(
        go.Bar(
            y=x["Claim Type"],
            x=x["State"],
            name="State",
            orientation="h",
            marker_color=COLORS["state"],
        )
    )

    fig.add_trace(
        go.Bar(
            y=x["Claim Type"],
            x=x["National"],
            name="National",
            orientation="h",
            marker_color=COLORS["national"],
        )
    )

    fig.update_layout(
        title=f"Claim-type spending — {phase}",
        barmode="group",
        xaxis_title="Average spending per episode",
        template="plotly_white",
        height=550,
    )

    fig.show()

In [ ]:
chart_claim_mix(
    "10001",
    "After discharge"
)

In [ ]:
def chart_divergence(
    facility_id,
    phase="After discharge"
):

    x = get_phase(
        facility_id,
        phase
    )

    x = x[
        x["Claim Type"] != "Total"
    ].copy()

    if x.empty:
        print("No data.")
        return

    x = (
        x
        .groupby("Claim Type", as_index=False)
        .agg(
            Facility=("Hospital", "sum"),
            State=("StateBenchmark", "sum"),
            National=("NationalBenchmark", "sum"),
        )
    )

    x["Facility - State"] = (
        x["Facility"] - x["State"]
    )

    x["Facility - National"] = (
        x["Facility"] - x["National"]
    )

    x = x.sort_values(
        "Facility - State"
    )

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            y=x["Claim Type"],
            x=x["Facility - State"],
            name="Facility − State",
            orientation="h",
            marker_color=[
                COLORS["negative"]
                if v < 0
                else COLORS["positive"]
                for v in x["Facility - State"]
            ],
        )
    )

    fig.add_trace(
        go.Bar(
            y=x["Claim Type"],
            x=x["Facility - National"],
            name="Facility − National",
            orientation="h",
            marker_color=COLORS["hospital"],
            opacity=0.45,
        )
    )

    fig.add_vline(
        x=0,
        line_width=2,
        line_color="#222222"
    )

    fig.update_layout(
        title=f"Where spending diverges — {phase}",
        xaxis_title="Facility spending minus benchmark",
        template="plotly_white",
        height=550,
    )

    fig.show()

In [ ]:
chart_divergence(
    "10001",
    "After discharge"
)

In [ ]:
def chart_inpatient_vs_snf(facility_id):

    x = get_phase(
        facility_id,
        "After discharge"
    )

    x = x[
        x["Claim Type"].isin([
            "Inpatient",
            "Skilled Nursing Facility",
        ])
    ]

    if x.empty:
        print("No data.")
        return

    x = (
        x
        .groupby("Claim Type", as_index=False)
        .agg(
            Facility=("Hospital", "sum"),
            State=("StateBenchmark", "sum"),
            National=("NationalBenchmark", "sum"),
        )
    )

    fig = go.Figure()

    for col, label, color in [
        ("Facility", "Facility", COLORS["hospital"]),
        ("State", "State", COLORS["state"]),
        ("National", "National", COLORS["national"]),
    ]:

        fig.add_trace(
            go.Bar(
                x=x["Claim Type"],
                y=x[col],
                name=label,
                marker_color=color,
            )
        )

    fig.update_layout(
        title="Post-discharge inpatient vs skilled nursing spending",
        barmode="group",
        yaxis_title="Average spending per episode",
        template="plotly_white",
        height=500,
    )

    fig.show()


In [ ]:
chart_inpatient_vs_snf("10001")

In [ ]:
def facility_benchmark_table():

    x = agg[
        (agg["Phase"] == "Complete episode")
        & (agg["Claim Type"] == "Total")
    ].copy()

    x["Facility vs State %"] = (
        (
            x["Hospital"]
            - x["StateBenchmark"]
        )
        / x["StateBenchmark"]
        * 100
    )

    x["Facility vs National %"] = (
        (
            x["Hospital"]
            - x["NationalBenchmark"]
        )
        / x["NationalBenchmark"]
        * 100
    )

    return x.sort_values(
        "Hospital",
        ascending=False
    )


benchmark = facility_benchmark_table()

display(
    benchmark[
        [
            "Facility Name",
            "Facility ID",
            "State",
            "Hospital",
            "StateBenchmark",
            "NationalBenchmark",
            "Facility vs State %",
            "Facility vs National %",
        ]
    ].head(20)
)

,Facility Name,Facility ID,State,Hospital,StateBenchmark,NationalBenchmark,Facility vs State %,Facility vs National %
61482,ORTHOCOLORADO HOSP AT ST ANTHONY MED CAMPUS,60124,CO,45654,28682,27477,59.173000,66.153510
24324,NEBRASKA SPINE HOSPITAL LLC,280133,NE,44693,27954,27477,59.880518,62.656040
57918,GREATER EL MONTE COMMUNITY HOSPITAL,50738,CA,44537,28825,27477,54.508239,62.088292
62032,CHI ST LUKES LAKESIDE HOSPITAL,670059,TX,43956,29550,27477,48.751269,59.973796
36644,OKLAHOMA SPINE HOSPITAL,370206,OK,43845,28175,27477,55.616681,59.569822
43288,HENDERSON COUNTY COMMUNITY HOSPITAL,440008,TN,43600,27845,27477,56.581074,58.678167
36556,NORTHWEST SURGICAL HOSPITAL,370192,OK,42912,28175,27477,52.305235,56.174255
7692,NORTHWEST SPECIALTY HOSPITAL,130066,ID,42088,27191,27477,54.786510,53.175383
20012,KARMANOS CANCER CENTER,230297,MI,41602,25972,27477,60.180194,51.406631
16294,THE SPINE HOSPITAL OF LOUISIANA,190266,LA,41530,28978,27477,43.315619,51.144594


In [ ]:
def chart_facility_distribution(
    state=None,
    top_n=20
):

    x = facility_benchmark_table()

    if state is not None:
        x = x[
            x["State"] == state
        ]

    x = x.head(top_n)

    x = x.sort_values(
        "Hospital"
    )

    fig = go.Figure(
        go.Bar(
            y=x["Facility Name"],
            x=x["Hospital"],
            orientation="h",
            marker_color=COLORS["hospital"],
            customdata=np.column_stack([
                x["State"],
                x["Facility ID"],
            ]),
            hovertemplate=(
                "<b>%{y}</b>"
                "<br>State: %{customdata[0]}"
                "<br>Facility ID: %{customdata[1]}"
                "<br>Spending: %{x:$,.0f}"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title="Highest complete-episode spending facilities",
        xaxis_title="Average spending per episode",
        template="plotly_white",
        height=650,
    )

    fig.show()

In [ ]:
chart_facility_distribution(top_n=20)

In [ ]:
# ============================================================
# INTERACTIVE FACILITY EXPLORER
# ============================================================

facility_lookup = (
    df[
        [
            "Facility ID",
            "Facility Name",
            "State",
        ]
    ]
    .drop_duplicates()
    .sort_values([
        "State",
        "Facility Name"
    ])
)

facility_options = [
    (
        f"{row['Facility Name']} | "
        f"{row['State']} | "
        f"ID {row['Facility ID']}",
        row["Facility ID"]
    )
    for _, row in facility_lookup.iterrows()
]


facility_widget = widgets.Dropdown(
    options=facility_options,
    description="Facility:",
    layout=widgets.Layout(
        width="700px"
    ),
)


phase_widget = widgets.Dropdown(
    options=[
        "Before admission",
        "Index admission",
        "After discharge",
    ],
    value="After discharge",
    description="Phase:",
    layout=widgets.Layout(
        width="450px"
    ),
)


output = widgets.Output()


def interactive_dashboard(
    facility_id,
    phase
):

    with output:

        clear_output(wait=True)

        info = facility_info(
            facility_id
        )

        if info is None:
            print("Facility not found.")
            return

        print(
            f"{info['name']} "
            f"({info['state']}) "
            f"— Facility ID {info['id']}"
        )

        print()

        chart_total_episode(
            facility_id
        )

        chart_episode_phases(
            facility_id
        )

        chart_claim_mix(
            facility_id,
            phase
        )

        chart_divergence(
            facility_id,
            phase
        )

        if phase == "After discharge":

            chart_inpatient_vs_snf(
                facility_id
            )


def update_dashboard(
    change=None
):

    interactive_dashboard(
        facility_widget.value,
        phase_widget.value
    )


facility_widget.observe(
    update_dashboard,
    names="value"
)

phase_widget.observe(
    update_dashboard,
    names="value"
)


display(
    widgets.VBox([
        facility_widget,
        phase_widget,
        output,
    ])
)

update_dashboard()

In [ ]:
def narrative_for_facility(
    facility_id
):

    info = facility_info(
        facility_id
    )

    total = get_complete_episode(
        facility_id
    )

    if info is None or total.empty:
        return "No data available."

    total = total[
        total["Claim Type"] == "Total"
    ].iloc[0]

    h = total["Hospital"]
    s = total["StateBenchmark"]
    n = total["NationalBenchmark"]

    vs_state = (
        (h - s) / s * 100
        if s != 0
        else np.nan
    )

    vs_nat = (
        (h - n) / n * 100
        if n != 0
        else np.nan
    )

    post = get_phase(
        facility_id,
        "After discharge"
    )

    post = (
        post
        .groupby("Claim Type")
        .agg(
            Facility=("Hospital", "sum"),
            State=("StateBenchmark", "sum"),
            National=("NationalBenchmark", "sum"),
        )
    )

    lines = []

    lines.append(
        f"### {info['name']}"
    )

    lines.append(
        f"Complete episode spending is "
        f"**{money(h)} per episode**, compared with "
        f"**{money(s)}** for the state and "
        f"**{money(n)}** nationally."
    )

    lines.append(
        f"That places the facility "
        f"**{vs_state:+.1f}% vs the state** and "
        f"**{vs_nat:+.1f}% vs the national benchmark**."
    )

    if "Inpatient" in post.index:

        inpatient = post.loc[
            "Inpatient"
        ]

        lines.append(
            f"After discharge, inpatient spending is "
            f"**{money(inpatient['Facility'])}**, "
            f"versus {money(inpatient['State'])} "
            f"for the state and "
            f"{money(inpatient['National'])} nationally."
        )

    if "Skilled Nursing Facility" in post.index:

        snf = post.loc[
            "Skilled Nursing Facility"
        ]

        lines.append(
            f"SNF spending is "
            f"**{money(snf['Facility'])}**, "
            f"versus {money(snf['State'])} "
            f"for the state and "
            f"{money(snf['National'])} nationally."
        )

    return "\n\n".join(lines)


print(
    narrative_for_facility("10001")
)

### SOUTHEAST HEALTH MEDICAL CENTER

Complete episode spending is **$26,979 per episode**, compared with **$26,206** for the state and **$27,477** nationally.

That places the facility **+2.9% vs the state** and **-1.8% vs the national benchmark**.

After discharge, inpatient spending is **$5,754**, versus $4,559 for the state and $4,316 nationally.

SNF spending is **$2,834**, versus $3,156 for the state and $4,194 nationally.


60K × 13 raw data
       │
       ▼
   clean once
       │
       ▼
 facility/phase/claim aggregation
       │
       ▼
    ~manageable
 analytical table
       │
       ├── episode benchmark
       ├── phase comparison
       ├── claim mix
       ├── benchmark divergence
       ├── inpatient vs SNF
       └── facility benchmarking